In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

In [2]:
# Load dataset
df = pd.read_csv("g2f_2023_phenotypic_clean_data.csv", encoding="latin1", low_memory=False)

print("Shape of dataset:", df.shape)
df.head()

Shape of dataset: (19434, 39)


,Year,Field-Location,State,City,Plot length (center-center in feet),Plot area (ft2),Alley length (in inches),Row spacing (in inches),Rows per plot,# Seed per plot,...,Root Lodging [# of plants],Stalk Lodging [# of plants],Grain Moisture [%],Test Weight [lbs],Plot Weight [lbs],Grain Yield (bu/A),Plot Discarded [enter 'yes' or blank],Comments,Filler,Snap [# of plants]
0,2023,COH1,CO,Fort Collins,20.0,85.0,36,30,2,80,...,NaN,NaN,25.1,51.3,26.9,218.201789,NaN,Full irrigation,NaN,NaN
1,2023,COH1,CO,Fort Collins,20.0,85.0,36,30,2,80,...,NaN,NaN,25.5,48.9,24.8,200.093123,NaN,Full irrigation,NaN,NaN
2,2023,COH1,CO,Fort Collins,20.0,85.0,36,30,2,80,...,NaN,NaN,16.2,56.7,28.8,261.372996,NaN,Full irrigation,NaN,NaN
3,2023,COH1,CO,Fort Collins,20.0,85.0,36,30,2,80,...,NaN,NaN,20.1,52.2,24.1,208.539358,NaN,Full irrigation,NaN,NaN
4,2023,COH1,CO,Fort Collins,20.0,85.0,36,30,2,80,...,NaN,NaN,28.0,48.4,27.4,213.652151,NaN,Full irrigation,NaN,NaN


In [3]:
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19434 entries, 0 to 19433
Data columns (total 39 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Year                                   19434 non-null  int64  
 1   Field-Location                         19434 non-null  object 
 2   State                                  19434 non-null  object 
 3   City                                   19434 non-null  object 
 4   Plot length (center-center in feet)    19434 non-null  float64
 5   Plot area (ft2)                        19434 non-null  float64
 6   Alley length (in inches)               19434 non-null  int64  
 7   Row spacing (in inches)                19434 non-null  int64  
 8   Rows per plot                          19434 non-null  int64  
 9   # Seed per plot                        19434 non-null  int64  
 10  Experiment                             19434 non-null  object 
 11  Pe

In [4]:
df.isna().sum().sort_values(ascending=False).head(20)

Filler                                   19434
Plot Discarded [enter 'yes' or blank]    19296
Comments                                 18793
Snap [# of plants]                       17697
Family                                   15447
Stand Count [# of plants]                 5252
Silking [days]                            4302
Silking [MM/DD/YY]                        4229
Anthesis [days]                           4141
Anthesis [MM/DD/YY]                       4099
Tester                                    3993
Stalk Lodging [# of plants]               2978
Root Lodging [# of plants]                2975
Test Weight [lbs]                         2308
Ear Height [cm]                           1801
Plant Height [cm]                         1790
Grain Moisture [%]                         661
Plot Weight [lbs]                          546
Grain Yield (bu/A)                         546
Date Plot Harvested [MM/DD/YY]             181
dtype: int64

In [5]:
# Target column
target = "Grain Yield (bu/A)"

# Drop rows with missing target
df = df.dropna(subset=[target]).copy()

# Columns that are likely IDs, comments, or mostly empty / not useful
drop_cols = [
    "Filler",
    "Comments",
    "Plot Discarded [enter 'yes' or blank]",
    "Plot_ID",
    "Plot",
    "Range",
    "Pass"
]

# Drop if they exist
drop_cols = [col for col in drop_cols if col in df.columns]
df = df.drop(columns=drop_cols)

print("Shape after dropping rows with missing target and unnecessary columns:", df.shape)

Shape after dropping rows with missing target and unnecessary columns: (18888, 32)


In [6]:
# Keep predictors except target
X = df.drop(columns=[target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (18888, 31)
y shape: (18888,)


In [7]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("\nCategorical features:", categorical_features)

Numeric features: ['Year', 'Plot length (center-center in feet)', 'Plot area (ft2)', 'Alley length (in inches)', 'Row spacing (in inches)', 'Rows per plot', '# Seed per plot', 'Replicate', 'Block', 'Anthesis [days]', 'Silking [days]', 'Plant Height [cm]', 'Ear Height [cm]', 'Stand Count [# of plants]', 'Root Lodging [# of plants]', 'Stalk Lodging [# of plants]', 'Grain Moisture [%]', 'Test Weight [lbs]', 'Plot Weight [lbs]', 'Snap [# of plants]']

Categorical features: ['Field-Location', 'State', 'City', 'Experiment', 'Pedigree', 'Family', 'Tester', 'Date Plot Planted [MM/DD/YY]', 'Date Plot Harvested [MM/DD/YY]', 'Anthesis [MM/DD/YY]', 'Silking [MM/DD/YY]']


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (15110, 31)
Test set: (3778, 31)


In [9]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [10]:
ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

In [11]:
ridge_param_grid = {
    "model__alpha": [0.1, 1.0, 10.0, 50.0, 100.0]
}

ridge_grid = GridSearchCV(
    ridge_pipeline,
    ridge_param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)

print("Best Ridge parameters:", ridge_grid.best_params_)
print("Best Ridge CV RMSE:", -ridge_grid.best_score_)

Best Ridge parameters: {'model__alpha': 1.0}
Best Ridge CV RMSE: 5.57324259958483


In [12]:
print(y.describe())

count    18888.000000
mean       160.392307
std         55.635837
min          6.821534
25%        121.888371
50%        164.638977
75%        199.271023
max        299.528944
Name: Grain Yield (bu/A), dtype: float64


In [13]:
rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best Random Forest parameters:", rf_grid.best_params_)
print("Best Random Forest CV RMSE:", -rf_grid.best_score_)

Best Random Forest parameters: {'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best Random Forest CV RMSE: 2.500952912042879
